# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainkhan006/Flyrank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Ranking Signal Analysis. I am framing it as scoring, not as “train a model.” I want to see which safe search and content numbers travel with visibility and clicks before I rank pages or train anything. The numbers are mixed together, so scoring associations is more honest than a one liner if-statement. The output is a signal report a content strategist can brief from: watch these associations on this snapshot; do not over-read those.

It is not classification as a yes/no label would push this toward ranking pages, and a current-window “declining” flag is a rule I defined, not a later observed outcome.
It is also not clustering as grouping pages into types is a different lane. Finally, it is also not ranking pages as ranking which page to edit first is Lane 2 (refresh queue). If I rank anything, it is which signals to watch, and scoring those associations is the cleaner name for that.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I will score or predict observed impressions_90d (visibility) and observed clicks_90d. Both already exist on each page row. Scoring those two outcomes is how I see which safe search and content numbers travel with being shown and being clicked. No new yes/no label will be created.

This label comes from measured outcomes on this starter snapshot (trailing 90 days), not from a rule I wrote. Because features and these outcomes sit in the same window, this is association on this slice, not “what will happen next month.”

I will not use trend_direction / a “declining” flag (trend_direction == "down") as a defined-rule proxy. That is a current-window bucket, a rule, not a later observed outcome. A scorer trained on that would learn my rule, not the world. Another proxy that will not be treated as a target is raw CTR. CTR mixes clicks with impressions and mostly follows position. Scoring CTR as if it were “quality” would over-read a confound. I keep impressions and clicks separate.

The output supports a content strategist’s brief of watching these associations on this snapshot, and not to over-read those. The scores feed the signal report.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The one metric I will defend is Spearman rank correlation between a candidate safe signal and the two observed outcomes, impressions_90d and clicks_90d. Spearman is a scoring metric for association (do the ranks travel together on this snapshot or not?). It is not Precision@50 on a page queue, and it is not a p-value. With 30,000 pages, a tiny correlation can look significant by chance, that is not enough to brief a strategist.

What "good” indicates for the brief is to not over-read (floor). Keyword search_volume vs impressions_90d was about 0.001 (Pearson) on this slice, a near-zero, failed one-line rule. A signal does not earn “watch” just by beating 0.001; that bar is too low by itself. When I code, I will report Spearman for that same pair so the floor uses this section’s metric. The signal’s Spearman with impressions and/or clicks is clearly stronger than that near-zero floor, and the sign (direction) repeats across most of the 32 clients, not only in the pooled table or in one lucky client. What is not good is a pooled Spearman that disappears or flips when you look client by client; or treating raw CTR or a declining flag as the thing I correlate against.
This number supports a decision-support brief on this snapshot: watch associations that clear that bar; do not over-read the rest. It does not mean the signal is a Google ranking factor, and it does not mean an edit will move impressions or clicks.

On this slice, Spearman for search_volume vs impressions_90d is −0.029 (Pearson 0.001; 27,532 rows with both values). vs clicks_90d it is −0.068. Near zero on ranks and on a straight line. A signal does not earn “watch” by beating this.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row in this slice is one page (one content item). It is not a client, not a day, and not a keyword. I loaded only the starter file: 30,000 pages across 32 clients. content_id and client_id are pseudonyms for grouping, not features and not real names or URLs.

The target sketch is the two observed columns I already named: impressions_90d (visibility) and clicks_90d. Both already exist on each row. I am not building a yes/no label and I am not using trend_direction as a target.

Signals and these two outcomes share the same trailing 90-day window, so this dataframe supports association on this snapshot, not “what happens next month.”

After the table I print Spearman for keyword search_volume vs those outcomes so the Section 3 floor uses this metric. On this slice: Spearman vs impressions_90d is −0.029 (Pearson 0.001; 27,532 rows with both values). vs clicks_90d it is −0.068. Near zero on ranks and on a straight line. That is the do-not-over-read floor, not a Google ranking factor, and not “search volume hurts visibility.”

In [1]:
import os
import sys
import subprocess
import pandas as pd

inColab = "google.colab" in sys.modules
repoUrl = "https://github.com/zainkhan006/Flyrank-ML"
repoDir = "Flyrank-ML"

if(inColab):
    if(not os.path.isdir("data/raw")):
        if(not os.path.isdir(repoDir)):
            subprocess.run(["git", "clone", "--depth", "1", repoUrl, repoDir], check=True)
        os.chdir(repoDir)

pages = pd.read_csv("data/raw/content_refresh_anonymized.csv")

nPages = len(pages)
nClients = pages["client_id"].nunique()

print("one row is one page")
print("pages in this slice:", nPages)
print("clients in this slice:", nClients)

previewCols = [
    "content_type",
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "word_count",
]
print("preview of safe columns (not a target recipe):")
display(pages[previewCols].head())

print("target sketch — observed visibility and clicks, already on each row:")
display(pages[["impressions_90d", "clicks_90d"]].head())
print(pages[["impressions_90d", "clicks_90d"]].describe())

volumeVsImpressionsPearson = pages["search_volume"].corr(pages["impressions_90d"], method="pearson")
volumeVsImpressionsSpearman = pages["search_volume"].corr(pages["impressions_90d"], method="spearman")
volumeVsClicksSpearman = pages["search_volume"].corr(pages["clicks_90d"], method="spearman")
nForSpearman = pages[["search_volume", "impressions_90d"]].dropna().shape[0]

print("rows used for search_volume correlations (both values present):", nForSpearman)
print("Pearson, search_volume vs impressions_90d:", round(volumeVsImpressionsPearson, 3))
print("Spearman, search_volume vs impressions_90d:", round(volumeVsImpressionsSpearman, 3))
print("Spearman, search_volume vs clicks_90d:", round(volumeVsClicksSpearman, 3))
print("near zero is the do-not-over-read floor, not a ranking-factor claim")

one row is one page
pages in this slice: 30000
clients in this slice: 32
preview of safe columns (not a target recipe):


,content_type,search_volume,impressions_90d,clicks_90d,avg_position,word_count
0,keyword article,10.0,3803,29,10.6,3221.0
1,keyword article,90.0,15320,7,20.3,2481.0
2,keyword article,0.0,12581,11,36.5,3515.0
3,keyword article,10.0,11751,58,6.2,NaN
4,keyword article,0.0,19140,24,44.0,2803.0


target sketch — observed visibility and clicks, already on each row:


,impressions_90d,clicks_90d
0,3803,29
1,15320,7
2,12581,11
3,11751,58
4,19140,24


       impressions_90d    clicks_90d
count     30000.000000  30000.000000
mean       5200.366300     16.097333
std       16838.019547     75.076958
min           1.000000      0.000000
25%          81.000000      0.000000
50%         731.000000      1.000000
75%        3615.250000      7.000000
max      517715.000000   4178.000000
rows used for search_volume correlations (both values present): 27532
Pearson, search_volume vs impressions_90d: 0.001
Spearman, search_volume vs impressions_90d: -0.029
Spearman, search_volume vs clicks_90d: -0.068
near zero is the do-not-over-read floor, not a ranking-factor claim


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A one-line if-statement is not enough here because the pattern is mixed, incomplete, and easy to over-read.

The obvious rule would be: high keyword search volume means high visibility. On this slice that rule fails. Spearman for search_volume vs impressions_90d is −0.029 (Pearson 0.001; 27,532 of 30,000 rows with both values). vs clicks_90d it is −0.068. Near zero on ranks and on a straight line. Briefing a strategist to “watch search volume” would send them after a number that barely travels with being shown or being clicked. A signal does not earn “watch” by beating this floor.

The other columns sit on the same row: position, type, word count, volume, impressions, clicks. Raw CTR looks like quality, but it mixes clicks with impressions and mostly follows position, so a rule on CTR would treat a confound as a signal. Features and outcomes share the same trailing 90-day window, so a rule cannot honestly say what happens next. This slice is 32 clients: a pooled cutoff can look fine in one table and disappear or flip on some sites. I have not claimed a client-by-client result yet; I am saying a single if-statement cannot see that risk.

Scoring several safe numbers against observed impressions_90d and clicks_90d is how I map that tangle without pretending one cutoff is the story. The output is still a signal report for a content strategist: watch associations that clear the bar; do not over-read the rest. It is not a page-refresh queue, not a Google ranking factor, and not a promise that an edit will move traffic.

In [2]:
print("why a one-line rule fails on this slice (same grain as Section 4):")
print("pages with no search_volume (the rule cannot even fire):", int(pages["search_volume"].isna().sum()))
print("Spearman, search_volume vs impressions_90d:", round(pages["search_volume"].corr(pages["impressions_90d"], method="spearman"), 3))
print("Spearman, search_volume vs clicks_90d:", round(pages["search_volume"].corr(pages["clicks_90d"], method="spearman"), 3))
print("near zero is the do-not-over-read floor, not a ranking-factor claim")

why a one-line rule fails on this slice (same grain as Section 4):
pages with no search_volume (the rule cannot even fire): 2468
Spearman, search_volume vs impressions_90d: -0.029
Spearman, search_volume vs clicks_90d: -0.068
near zero is the do-not-over-read floor, not a ranking-factor claim


## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.